# MELA — model development and validation

Develops the MELA ensemble (XGBoost, AdaBoost, CatBoost; positive when at least 2 of 3 models
vote positive) on 80% of the derivation cohort and evaluates it, with every model and
threshold locked, on the 20% held-out test and on the external validation cohort (PRESC1SE).
LightGBM, Logistic Regression, Random Forest and TabPFN are developed with the same procedure
and reported alongside, but never enter the vote.

- Thresholds are locked on development data and applied literally (`score >= threshold`).
- Mean imputation and standardisation are fitted inside each training partition.
- Four symptom variables are zero-coded when missing, as a fixed clinical rule.
- 95% CIs are exact Clopper–Pearson intervals.

Outputs: aggregate tables in `results/locked/`, per-patient predictions in
`results/predictions/` (git-ignored), locked models in `models/` (used by the inference notebooks).

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Dict, Iterable, Tuple

import joblib
import numpy as np
import pandas as pd
from scipy.stats import beta

from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, matthews_corrcoef, roc_curve
from sklearn.metrics._scorer import make_scorer
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from tabpfn import TabPFNClassifier

## 1. Configuration

In [2]:
RANDOM_STATE = 42
MIN_SENSITIVITY = 0.98
TEST_SIZE = 0.20
N_SPLITS_OUTER = 5
N_SPLITS_INNER = 5

# Choice among the five outer-fold candidates, and source of the threshold.
SELECTION_CRITERION = "train"   # {'train', 'fold', 'oof'}
THRESHOLD_SOURCE = "fold"       # {'fold', 'oof'}

# Missing = absent for these binary symptoms (fixed recoding, not a learned imputation).
FIXED_ZERO_COLS = ("post_chestpain", "abd_pain", "lumb_pain", "syncope")

# MELA voting ensemble: positive (rule-in) if >= N_VOTES_POSITIVE of the three models are positive.
MODEL_NAMES = ("XGBoost", "AdaBoost", "CatBoost")
N_VOTES_POSITIVE = 2

# Developed and reported alongside the ensemble, never part of the vote.
EXTRA_MODEL_NAMES = ("TabPFN", "LightGBM", "LogisticRegression", "RandomForest")

# Imposed sensitivities of the specificity table (section 8).
SENSITIVITY_TARGETS = (0.97, 0.98, 0.99)

REPO_ROOT = Path("..")
DATA_DIR = REPO_ROOT / "data"
PROFUNDUS_PATH = DATA_DIR / "profundus_modifiedCorrected.csv"
ADVISED_PATH = DATA_DIR / "ADVISED_modified.csv"
PRESC1SE_PATH = DATA_DIR / "master_PRESC1SE_Flavio.xlsx"

OUTPUT_DIR = REPO_ROOT / "results" / "locked"            # aggregate tables
PREDICTIONS_DIR = REPO_ROOT / "results" / "predictions"  # per-patient rows (git-ignored)
MODELS_DIR = REPO_ROOT / "models"                        # locked pipelines

ID_COL = "patient_id"   # original ID of each cohort

## 2. Data harmonisation

In [3]:
def fixed_zero_recode(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in FIXED_ZERO_COLS:
        if col in out.columns:
            out[col] = out[col].fillna(0.0)
    return out


def harmonise_smoking_external(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    mapping_smoke = {
        "never smoked": 0,
        "former smoker": 1,
        "unknown": 0,
        "active smoker": 1,
        0: 0,
        "0": 0,
    }
    if "smoke" in out.columns:
        out["smoke"] = out["smoke"].map(mapping_smoke).fillna(0).astype(int)
    return out


def require_columns(df: pd.DataFrame, columns: Iterable[str], cohort: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"{cohort}: missing required columns: {missing}")


def load_cohorts():
    """Every cohort is indexed by its original patient ID (ID_COL)."""
    data_dsm = pd.read_csv(PROFUNDUS_PATH, index_col="record_id").dropna(subset=["ddimer_value"])
    data_adv = pd.read_csv(ADVISED_PATH, index_col="Uniq_ID").dropna(subset=["ddimer_value"])

    data_merged = fixed_zero_recode(pd.concat([data_dsm, data_adv], axis=0))
    data_merged.index.name = ID_COL
    if "final_dgn" not in data_merged.columns:
        raise ValueError("Derivation cohort: target 'final_dgn' not found")

    features = [c for c in data_merged.columns if c != "final_dgn"]
    X_merged = data_merged.loc[:, features].copy()
    y_merged = data_merged["final_dgn"].astype(int).to_numpy()

    data_ext = pd.read_excel(PRESC1SE_PATH).set_index("record_id")
    data_ext.index.name = ID_COL
    data_ext = harmonise_smoking_external(data_ext)
    data_ext = data_ext.dropna(subset=["final_dgn"]).copy()
    data_ext = fixed_zero_recode(data_ext)
    data_ext["ddimer_value"] = np.log2(data_ext["ddimer_value"])
    data_ext = data_ext.replace([np.inf, -np.inf], np.nan)
    data_ext = data_ext.dropna(subset=["ddimer_value"]).copy()

    require_columns(data_ext, [*features, "final_dgn"], "External validation")
    X_ext = data_ext.loc[:, features].copy()
    y_ext = data_ext["final_dgn"].astype(int).to_numpy()

    print("Cohorts after harmonisation")
    for label, y in (("Derivation", y_merged), ("External validation", y_ext)):
        print(f"  {label:<22} n={len(y):5d}  positives={int(y.sum()):4d} ({y.mean():.2%})")

    return X_merged, y_merged, X_ext, y_ext, data_dsm.index

## 3. Preprocessing and metric utilities

In [4]:
def make_pipeline(estimator) -> Pipeline:
    """All learned preprocessing is fitted inside each training partition."""
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(missing_values=np.nan, strategy="mean")),
            ("scaler", StandardScaler()),
            ("clf", estimator),
        ]
    )


def operating_point(
    y_true: np.ndarray,
    y_score: np.ndarray,
    min_sens: float = MIN_SENSITIVITY,
) -> Tuple[float, float, float]:
    """Highest-specificity ROC point with sensitivity >= min_sens (ties: highest threshold)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_score, drop_intermediate=False)
    eligible = np.flatnonzero(tpr >= min_sens)
    if len(eligible) == 0:
        raise ValueError(f"No threshold reaches sensitivity >= {min_sens:.3f}")

    specificity = 1.0 - fpr
    best_spec = specificity[eligible].max()
    tied = eligible[np.isclose(specificity[eligible], best_spec)]
    best_idx = tied[np.argmax(thresholds[tied])]
    return float(tpr[best_idx]), float(specificity[best_idx]), float(thresholds[best_idx])


def clopper_pearson(k: int, n: int, alpha: float = 0.05) -> Tuple[float, float]:
    """Exact two-sided binomial confidence interval."""
    if n <= 0:
        return np.nan, np.nan
    lo = 0.0 if k == 0 else beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else beta.ppf(1 - alpha / 2, k + 1, n - k)
    return float(lo), float(hi)


def binary_metrics_from_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, float]:
    """Confusion-matrix metrics and exact 95% CIs."""
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    n_pos = tp + fn
    n_neg = tn + fp

    sens = tp / n_pos if n_pos else np.nan
    spec = tn / n_neg if n_neg else np.nan
    sens_lo, sens_hi = clopper_pearson(tp, n_pos)
    spec_lo, spec_hi = clopper_pearson(tn, n_neg)

    return {
        "n": int(len(y_true)),
        "positives": int(n_pos),
        "negatives": int(n_neg),
        "TP": int(tp),
        "FN": int(fn),
        "TN": int(tn),
        "FP": int(fp),
        "Sensitivity": float(sens),
        "Sensitivity_CI_low": sens_lo,
        "Sensitivity_CI_high": sens_hi,
        "Specificity": float(spec),
        "Specificity_CI_low": spec_lo,
        "Specificity_CI_high": spec_hi,
    }


def evaluate_fixed_threshold(
    y_true: np.ndarray,
    y_score: np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    """Apply the locked threshold literally; do not re-enter roc_curve()."""
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    return binary_metrics_from_predictions(y_true, y_pred)

## 4. Model development on the 80% development set only

In [5]:
def develop_model(
    name: str,
    estimator,
    param_grid: dict,
    X_dev: pd.DataFrame,
    y_dev: np.ndarray,
    *,
    seed: int = RANDOM_STATE,
    n_jobs: int = -1,
):
    """Develop and lock one model using the development set only."""
    outer_cv = StratifiedKFold(
        n_splits=N_SPLITS_OUTER,
        shuffle=True,
        random_state=seed,
    )

    configs = []
    rows = []

    # Phase 1: inner GridSearchCV on each outer-training fold.
    for fold_id, (tr, te) in enumerate(outer_cv.split(X_dev, y_dev)):
        grid = GridSearchCV(
            estimator=make_pipeline(clone(estimator)),
            param_grid=param_grid,
            scoring=make_scorer(matthews_corrcoef),
            cv=StratifiedKFold(
                n_splits=N_SPLITS_INNER,
                shuffle=True,
                random_state=seed + fold_id + 1,
            ),
            n_jobs=n_jobs,
            refit=True,
        )
        grid.fit(X_dev.iloc[tr], y_dev[tr])

        fold_score = grid.best_estimator_.predict_proba(X_dev.iloc[te])[:, 1]
        fold_sens, fold_spec, fold_thr = operating_point(y_dev[te], fold_score)
        fold_pred = (fold_score >= fold_thr).astype(int)

        configs.append(dict(grid.best_params_))
        rows.append(
            {
                "Config": fold_id,
                "MCC_grid": float(grid.best_score_),
                "Fold_Sens": fold_sens,
                "Fold_Spec": fold_spec,
                "Fold_Threshold": fold_thr,
                "Fold_MCC_at_threshold": float(matthews_corrcoef(y_dev[te], fold_pred)),
            }
        )

    # Phase 2: refit each candidate configuration on the complete development set.
    fitted = []
    oof_scores = []
    for k, params in enumerate(configs):
        model = make_pipeline(clone(estimator)).set_params(**params)
        model.fit(X_dev, y_dev)
        fitted.append(model)

        train_score = model.predict_proba(X_dev)[:, 1]
        tr_sens, tr_spec, tr_thr = operating_point(y_dev, train_score)

        # Diagnostic OOF scores (not used for selection when SELECTION_CRITERION='train').
        oof_score = cross_val_predict(
            make_pipeline(clone(estimator)).set_params(**params),
            X_dev,
            y_dev,
            cv=outer_cv,
            method="predict_proba",
            n_jobs=n_jobs,
        )[:, 1]
        oof_sens, oof_spec, oof_thr = operating_point(y_dev, oof_score)
        oof_scores.append(oof_score)

        rows[k].update(
            {
                "Train_Sens": tr_sens,
                "Train_Spec": tr_spec,
                "Train_Threshold": tr_thr,
                "OOF_Sens": oof_sens,
                "OOF_Spec": oof_spec,
                "OOF_Threshold": oof_thr,
            }
        )

    table = pd.DataFrame(rows)

    # Phase 3: select among the five candidate configurations.
    criteria = {
        "train": ("Train_Sens", "Train_Spec", "Fold_Spec"),
        "fold": ("Fold_Sens", "Fold_Spec", "OOF_Spec"),
        "oof": ("OOF_Sens", "OOF_Spec", "Fold_Spec"),
    }
    if SELECTION_CRITERION not in criteria:
        raise ValueError(f"Unknown SELECTION_CRITERION={SELECTION_CRITERION!r}")

    sens_col, spec_col, tie_col = criteria[SELECTION_CRITERION]
    eligible = table.loc[table[sens_col] >= MIN_SENSITIVITY].copy()
    if eligible.empty:
        raise ValueError(
            f"{name}: no candidate reaches {sens_col} >= {MIN_SENSITIVITY:.3f}"
        )

    best = eligible.sort_values(
        [spec_col, tie_col, "MCC_grid"],
        ascending=[False, False, False],
    ).iloc[0]
    best_k = int(best["Config"])
    best_params = configs[best_k]

    # Folds that selected the same hyperparameters share the median fold threshold.
    twin_ids = [i for i, p in enumerate(configs) if p == best_params]

    if THRESHOLD_SOURCE == "oof":
        threshold = float(best["OOF_Threshold"])
        threshold_note = "OOF threshold on development"
    elif THRESHOLD_SOURCE == "fold":
        fold_thresholds = table.loc[
            table["Config"].isin(twin_ids), "Fold_Threshold"
        ].to_numpy(dtype=float)
        threshold = float(np.median(fold_thresholds))
        if len(fold_thresholds) > 1:
            threshold_note = (
                f"median fold threshold across {len(fold_thresholds)} identical "
                "hyperparameter configurations"
            )
        else:
            threshold_note = "outer-fold threshold"
    else:
        raise ValueError(f"Unknown THRESHOLD_SOURCE={THRESHOLD_SOURCE!r}")

    print(f"\n--- {name}: LOCKED ---")
    print(f"Selection criterion : {SELECTION_CRITERION}")
    print(f"Selected config     : {best_k}")
    print(f"Selected parameters : {best_params}")
    print(f"Threshold           : {threshold:.8g} ({threshold_note})")
    print(table.loc[table["Config"] == best_k].round(4).to_string(index=False))

    return fitted[best_k], threshold, table, best_params, oof_scores[best_k]

## 5. Candidate algorithms

In [6]:
def get_model_specs():
    """Grids of the hyperparameter study (first value of each list = configuration selected)."""
    specs = {
        "XGBoost": (
            xgb.XGBClassifier(random_state=RANDOM_STATE),
            {
                "clf__max_depth": [3, 2],
                "clf__learning_rate": [0.03, 0.01],
                "clf__n_estimators": [300, 100],
                "clf__min_child_weight": [10],
                "clf__subsample": [0.8],
                "clf__colsample_bytree": [0.5],
                "clf__reg_lambda": [10],
            },
            -1,
        ),
        "AdaBoost": (
            AdaBoostClassifier(random_state=RANDOM_STATE),
            {
                "clf__n_estimators": [200, 100],
                "clf__learning_rate": [0.3, 0.1],
            },
            -1,
        ),
        "CatBoost": (
            CatBoostClassifier(random_state=RANDOM_STATE, verbose=0),
            {
                "clf__iterations": [50, 25],
                "clf__learning_rate": [0.2, 0.1, 0.05],
                "clf__depth": [2],
            },
            -1,
        ),
        "LightGBM": (
            LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=1),
            {
                "clf__num_leaves": [15],
                "clf__learning_rate": [0.2, 0.1],
                "clf__n_estimators": [500, 100],
            },
            -1,
        ),
        "LogisticRegression": (
            LogisticRegression(max_iter=10000, random_state=RANDOM_STATE),
            {
                "clf__C": [10, 100],
                "clf__penalty": ["l2", "l1"],
                "clf__solver": ["liblinear"],
                "clf__class_weight": ["balanced"],
            },
            -1,
        ),
        "RandomForest": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
            {
                "clf__n_estimators": [600, 300],
                "clf__max_depth": [10, 5],
                "clf__min_samples_leaf": [1, 5],
                "clf__max_features": [0.5],
                "clf__criterion": ["gini"],
                "clf__class_weight": ["balanced_subsample"],
            },
            -1,
        ),
    }

    # TabPFN needs a licence token from the environment; without it, it is skipped.
    if not os.getenv("TABPFN_TOKEN"):
        print("TABPFN_TOKEN not set - skipping TabPFN.")
        return specs

    specs["TabPFN"] = (
        TabPFNClassifier(device="auto", random_state=RANDOM_STATE),
        {
            "clf__n_estimators": [1],
            "clf__inference_config": [{"SUBSAMPLE_SAMPLES": 300}, {"SUBSAMPLE_SAMPLES": 100}],
        },
        1,  # avoid loading several TabPFN instances on one GPU in parallel
    )
    return specs

## 6. Final locked evaluation

In [7]:
def evaluate_single_models(
    models: dict,
    thresholds: dict,
    cohorts: dict,
) -> pd.DataFrame:
    """Evaluate only after every model/threshold has been locked."""
    rows = []
    for name, model in models.items():
        threshold = thresholds[name]
        for cohort_name, (X, y) in cohorts.items():
            score = model.predict_proba(X)[:, 1]
            m = evaluate_fixed_threshold(y, score, threshold)
            rows.append(
                {
                    "Model": name,
                    "Cohort": cohort_name,
                    "Threshold": threshold,
                    **m,
                }
            )
    return pd.DataFrame(rows)


def evaluate_development_oof(
    oof_scores: dict,
    thresholds: dict,
    y_dev: np.ndarray,
) -> pd.DataFrame:
    """Out-of-fold performance on the development set at the locked thresholds.

    Cross-validated for model fitting but not for threshold selection (thresholds
    come from the development set); the held-out test is the clean estimate.
    """
    rows = []
    votes = []
    for name in MODEL_NAMES:
        pred = (oof_scores[name] >= thresholds[name]).astype(int)
        votes.append(pred)
        rows.append(
            {
                "Model": name,
                "Cohort": "Development (out-of-fold)",
                "Threshold": thresholds[name],
                **binary_metrics_from_predictions(y_dev, pred),
            }
        )

    ensemble = (np.vstack(votes).sum(axis=0) >= N_VOTES_POSITIVE).astype(int)
    rows.append(
        {
            "Model": f"{len(MODEL_NAMES)}-model majority vote (>={N_VOTES_POSITIVE}/{len(MODEL_NAMES)})",
            "Cohort": "Development (out-of-fold)",
            "Threshold": np.nan,
            **binary_metrics_from_predictions(y_dev, ensemble),
        }
    )
    return pd.DataFrame(rows)


def voting_predict(
    X: pd.DataFrame,
    models: dict,
    thresholds: dict,
) -> Tuple[np.ndarray, pd.DataFrame]:
    """Hard majority vote of the MODEL_NAMES models."""
    vote_dict = {}
    for name in MODEL_NAMES:
        score = models[name].predict_proba(X)[:, 1]
        vote_dict[name] = (score >= thresholds[name]).astype(int)

    votes = pd.DataFrame(vote_dict, index=X.index)
    final_pred = (votes.sum(axis=1) >= N_VOTES_POSITIVE).astype(int).to_numpy()
    return final_pred, votes


def evaluate_voting_ensemble(
    models: dict,
    thresholds: dict,
    cohorts: dict,
) -> Tuple[pd.DataFrame, dict, dict]:
    rows = []
    predictions = {}
    vote_tables = {}

    for cohort_name, (X, y) in cohorts.items():
        pred, votes = voting_predict(X, models, thresholds)
        m = binary_metrics_from_predictions(y, pred)
        rows.append(
            {
                "Model": f"{len(MODEL_NAMES)}-model majority vote "
                         f"(>={N_VOTES_POSITIVE}/{len(MODEL_NAMES)})",
                "Cohort": cohort_name,
                **m,
            }
        )
        predictions[cohort_name] = pred
        vote_tables[cohort_name] = votes

    return pd.DataFrame(rows), predictions, vote_tables


def pretty_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Sensitivity_95CI"] = out.apply(
        lambda r: (
            f"{r['Sensitivity']:.3f} "
            f"({r['Sensitivity_CI_low']:.3f}-{r['Sensitivity_CI_high']:.3f})"
        ),
        axis=1,
    )
    out["Specificity_95CI"] = out.apply(
        lambda r: (
            f"{r['Specificity']:.3f} "
            f"({r['Specificity_CI_low']:.3f}-{r['Specificity_CI_high']:.3f})"
        ),
        axis=1,
    )
    keep = [
        "Model",
        "Cohort",
        "n",
        "positives",
        "TP",
        "FN",
        "TN",
        "FP",
        "Sensitivity_95CI",
        "Specificity_95CI",
    ]
    return out.loc[:, keep]

## 7. Main analysis

In [8]:
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

    X_all, y_all, X_ext, y_ext, profundus_ids = load_cohorts()

    # Held-out split before any learned preprocessing or model selection.
    X_dev, X_test, y_dev, y_test = train_test_split(
        X_all,
        y_all,
        test_size=TEST_SIZE,
        stratify=y_all,
        random_state=RANDOM_STATE,
    )

    print("\nInitial split")
    print(
        f"  Development   n={len(y_dev):5d}, positives={int(y_dev.sum()):4d} "
        f"({y_dev.mean():.2%})"
    )
    print(
        f"  HELD-OUT TEST n={len(y_test):5d}, positives={int(y_test.sum()):4d} "
        f"({y_test.mean():.2%})"
    )
    print(
        f"  Sensitivity resolution in held-out = {1 / y_test.sum():.2%} per positive case"
    )

    specs = get_model_specs()
    final_models = {}
    final_thresholds = {}
    development_tables = {}
    selected_params = {}
    development_oof = {}

    to_develop = list(MODEL_NAMES) + [n for n in EXTRA_MODEL_NAMES if n in specs]
    print(f"\nEnsemble members : {', '.join(MODEL_NAMES)} "
          f"(positive if >={N_VOTES_POSITIVE}/{len(MODEL_NAMES)})")
    extras = [n for n in to_develop if n not in MODEL_NAMES]
    print(f"Exploratory only : {', '.join(extras) if extras else 'none'}")

    for name in to_develop:
        estimator, grid, n_jobs = specs[name]
        model, threshold, table, params, oof_score = develop_model(
            name,
            estimator,
            grid,
            X_dev,
            y_dev,
            n_jobs=n_jobs,
        )
        final_models[name] = model
        final_thresholds[name] = threshold
        development_tables[name] = table
        selected_params[name] = params
        development_oof[name] = oof_score
        table.to_csv(OUTPUT_DIR / f"development_{name}.csv", index=False)

    # Models and thresholds are locked: only now touch held-out and external data.
    cohorts = {
        "HELD-OUT TEST (20%)": (X_test, y_test),
        "External validation (PRESC1SE)": (X_ext, y_ext),
    }

    single_results = evaluate_single_models(
        final_models,
        final_thresholds,
        cohorts,
    )
    ensemble_results, _, _ = evaluate_voting_ensemble(
        final_models,
        final_thresholds,
        cohorts,
    )

    dev_oof_results = evaluate_development_oof(
        development_oof,
        final_thresholds,
        y_dev,
    )

    print("\n=== DEVELOPMENT COHORT, CROSS-VALIDATED (out-of-fold) ===")
    print(pretty_metrics(dev_oof_results).to_string(index=False))

    print("\n=== LOCKED SINGLE-MODEL PERFORMANCE ===")
    print(pretty_metrics(single_results).to_string(index=False))

    print(f"\n=== FINAL {len(MODEL_NAMES)}-MODEL MAJORITY VOTE "
          f"(positive if >={N_VOTES_POSITIVE}/{len(MODEL_NAMES)}) ===")
    print(pretty_metrics(ensemble_results).to_string(index=False))

    dev_oof_results.to_csv(OUTPUT_DIR / "development_oof_results.csv", index=False)
    single_results.to_csv(OUTPUT_DIR / "single_model_locked_results.csv", index=False)
    ensemble_results.to_csv(OUTPUT_DIR / f"ensemble_{N_VOTES_POSITIVE}of{len(MODEL_NAMES)}_locked_results.csv", index=False)

    locked_summary = pd.DataFrame(
        [
            {
                "Model": name,
                "Threshold": final_thresholds[name],
                "SelectionCriterion": SELECTION_CRITERION,
                "ThresholdSource": THRESHOLD_SOURCE,
                "SelectedParameters": repr(selected_params[name]),
            }
            for name in final_models
        ]
    )
    locked_summary.to_csv(OUTPUT_DIR / "locked_models_and_thresholds.csv", index=False)

    print("\nSaved outputs in:", OUTPUT_DIR)

    return {
        "X_dev": X_dev, "y_dev": y_dev,
        "X_test": X_test, "y_test": y_test,
        "X_ext": X_ext, "y_ext": y_ext,
        "profundus_ids": profundus_ids,
        "models": final_models,
        "thresholds": final_thresholds,
        "development_oof": development_oof,
        "development_tables": development_tables,
        "dev_oof_results": dev_oof_results,
        "single_results": single_results,
        "ensemble_results": ensemble_results,
    }

if __name__ == "__main__":
    # TabPFN: export TABPFN_TOKEN in the shell before launching Jupyter, never in this file.
    results = main()

Cohorts after harmonisation
  Derivation             n= 4289  positives= 336 (7.83%)
  External validation    n= 1282  positives=  28 (2.18%)

Initial split
  Development   n= 3431, positives= 269 (7.84%)
  HELD-OUT TEST n=  858, positives=  67 (7.81%)
  Sensitivity resolution in held-out = 1.49% per positive case

Ensemble members : XGBoost, AdaBoost, CatBoost (positive if >=2/3)
Exploratory only : TabPFN, LightGBM, LogisticRegression, RandomForest



--- XGBoost: LOCKED ---
Selection criterion : train
Selected config     : 2
Selected parameters : {'clf__colsample_bytree': 0.5, 'clf__learning_rate': 0.03, 'clf__max_depth': 3, 'clf__min_child_weight': 10, 'clf__n_estimators': 300, 'clf__reg_lambda': 10, 'clf__subsample': 0.8}
Threshold           : 0.015033977 (median fold threshold across 3 identical hyperparameter configurations)
 Config  MCC_grid  Fold_Sens  Fold_Spec  Fold_Threshold  Fold_MCC_at_threshold  Train_Sens  Train_Spec  Train_Threshold  OOF_Sens  OOF_Spec  OOF_Threshold
      2    0.5372     0.9815     0.7358          0.0212                 0.4138      0.9814      0.6967           0.0195    0.9814     0.636          0.015



--- AdaBoost: LOCKED ---
Selection criterion : train
Selected config     : 1
Selected parameters : {'clf__learning_rate': 0.3, 'clf__n_estimators': 200}
Threshold           : 0.2867102 (median fold threshold across 5 identical hyperparameter configurations)
 Config  MCC_grid  Fold_Sens  Fold_Spec  Fold_Threshold  Fold_MCC_at_threshold  Train_Sens  Train_Spec  Train_Threshold  OOF_Sens  OOF_Spec  OOF_Threshold
      1    0.5575     0.9811     0.7915          0.3505                 0.4656      0.9814      0.7094           0.3103    0.9814     0.512         0.2646



--- CatBoost: LOCKED ---
Selection criterion : train
Selected config     : 1
Selected parameters : {'clf__depth': 2, 'clf__iterations': 50, 'clf__learning_rate': 0.2}
Threshold           : 0.016142942 (median fold threshold across 5 identical hyperparameter configurations)
 Config  MCC_grid  Fold_Sens  Fold_Spec  Fold_Threshold  Fold_MCC_at_threshold  Train_Sens  Train_Spec  Train_Threshold  OOF_Sens  OOF_Spec  OOF_Threshold
      1    0.5749     0.9811      0.793          0.0426                 0.4674      0.9814      0.6028           0.0136    0.9814    0.5927         0.0131



--- TabPFN: LOCKED ---
Selection criterion : train
Selected config     : 1
Selected parameters : {'clf__inference_config': {'SUBSAMPLE_SAMPLES': 300}, 'clf__n_estimators': 1}
Threshold           : 0.0017167043 (median fold threshold across 5 identical hyperparameter configurations)
 Config  MCC_grid  Fold_Sens  Fold_Spec  Fold_Threshold  Fold_MCC_at_threshold  Train_Sens  Train_Spec  Train_Threshold  OOF_Sens  OOF_Spec  OOF_Threshold
      1    0.5642     0.9811     0.7615          0.0293                 0.4344      0.9814      0.5601           0.0025    0.9814    0.4775         0.0009



--- LightGBM: LOCKED ---
Selection criterion : train
Selected config     : 0
Selected parameters : {'clf__learning_rate': 0.2, 'clf__n_estimators': 500, 'clf__num_leaves': 15}
Threshold           : 2.2451609e-11 (median fold threshold across 2 identical hyperparameter configurations)
 Config  MCC_grid  Fold_Sens  Fold_Spec  Fold_Threshold  Fold_MCC_at_threshold  Train_Sens  Train_Spec  Train_Threshold  OOF_Sens  OOF_Spec  OOF_Threshold
      0    0.5788     0.9815     0.3602             0.0                 0.1951      0.9814         1.0            0.998    0.9814    0.4279            0.0



--- LogisticRegression: LOCKED ---
Selection criterion : train
Selected config     : 4
Selected parameters : {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}
Threshold           : 0.10013778 (median fold threshold across 3 identical hyperparameter configurations)
 Config  MCC_grid  Fold_Sens  Fold_Spec  Fold_Threshold  Fold_MCC_at_threshold  Train_Sens  Train_Spec  Train_Threshold  OOF_Sens  OOF_Spec  OOF_Threshold
      4    0.4681     0.9815     0.7832          0.2632                 0.4602      0.9814      0.6553           0.1332    0.9814    0.4225          0.047



--- RandomForest: LOCKED ---
Selection criterion : train
Selected config     : 4
Selected parameters : {'clf__class_weight': 'balanced_subsample', 'clf__criterion': 'gini', 'clf__max_depth': 10, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 600}
Threshold           : 0.024686005 (median fold threshold across 2 identical hyperparameter configurations)
 Config  MCC_grid  Fold_Sens  Fold_Spec  Fold_Threshold  Fold_MCC_at_threshold  Train_Sens  Train_Spec  Train_Threshold  OOF_Sens  OOF_Spec  OOF_Threshold
      4      0.56     0.9815     0.7547          0.0491                 0.4313      0.9814       0.987           0.5774    0.9814    0.3738         0.0017



=== DEVELOPMENT COHORT, CROSS-VALIDATED (out-of-fold) ===
                        Model                    Cohort    n  positives  TP  FN   TN   FP    Sensitivity_95CI    Specificity_95CI
                      XGBoost Development (out-of-fold) 3431        269 264   5 2011 1151 0.981 (0.957-0.994) 0.636 (0.619-0.653)
                     AdaBoost Development (out-of-fold) 3431        269 261   8 1948 1214 0.970 (0.942-0.987) 0.616 (0.599-0.633)
                     CatBoost Development (out-of-fold) 3431        269 260   9 2018 1144 0.967 (0.937-0.985) 0.638 (0.621-0.655)
3-model majority vote (>=2/3) Development (out-of-fold) 3431        269 262   7 1994 1168 0.974 (0.947-0.989) 0.631 (0.614-0.647)

=== LOCKED SINGLE-MODEL PERFORMANCE ===
             Model                         Cohort    n  positives  TP  FN   TN   FP    Sensitivity_95CI    Specificity_95CI
           XGBoost            HELD-OUT TEST (20%)  858         67  66   1  509  282 0.985 (0.920-1.000) 0.643 (0.609-0.677)
  

## 8. Out-of-fold specificity at imposed sensitivity

Development set, out-of-fold predictions of each locked configuration. For every imposed sensitivity the
threshold is the highest-specificity point with sensitivity ≥ target on those predictions; 95% CI by the
exact Clopper–Pearson method.

In [9]:
ORDER = ["XGBoost", "CatBoost", "AdaBoost", "TabPFN", "LightGBM", "LogisticRegression", "RandomForest"]
LABEL = {"LogisticRegression": "Logistic Regression", "RandomForest": "Random Forest"}
ORDER = [m for m in ORDER if m in results["development_oof"]]

y_dev = results["y_dev"]
rows = []
for name in ORDER:
    oof = results["development_oof"][name]
    for target in SENSITIVITY_TARGETS:
        _, _, thr = operating_point(y_dev, oof, min_sens=target)
        pred = (oof >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_dev, pred, labels=[0, 1]).ravel()
        lo, hi = clopper_pearson(tn, tn + fp)
        rows.append({"Model": LABEL.get(name, name), "Target": target, "TP": tp, "FN": fn, "TN": tn, "FP": fp,
                     "Sensitivity": tp / (tp + fn), "Specificity": tn / (tn + fp), "CP_low": lo, "CP_high": hi})
spec_table = pd.DataFrame(rows)
spec_table.to_csv(OUTPUT_DIR / "oof_specificity_by_target.csv", index=False)

labels = [LABEL.get(m, m) for m in ORDER]
table = spec_table.pivot(index="Model", columns="Target", values="Specificity").loc[labels]
table.columns = [f"{t:.0%}" for t in table.columns]
table.index.name = "Modello"
print(f"Development set, out-of-fold (n={len(y_dev)}, {int(y_dev.sum())} AAS). Achieved sensitivity: "
      + " / ".join(f"{s:.3f}" for s in spec_table.groupby("Target")["Sensitivity"].first()))
print("\nLocked thresholds:", {k: round(v, 6) for k, v in results["thresholds"].items()})
table.round(3)

Development set, out-of-fold (n=3431, 269 AAS). Achieved sensitivity: 0.970 / 0.981 / 0.993

Locked thresholds: {'XGBoost': 0.015034, 'AdaBoost': 0.28671, 'CatBoost': 0.016143, 'TabPFN': 0.001717, 'LightGBM': 0.0, 'LogisticRegression': 0.100138, 'RandomForest': 0.024686}


,97%,98%,99%
Modello,,,
XGBoost,0.666,0.636,0.374
CatBoost,0.631,0.593,0.513
AdaBoost,0.616,0.512,0.139
TabPFN,0.609,0.478,0.328
LightGBM,0.470,0.428,0.257
Logistic Regression,0.591,0.423,0.295
Random Forest,0.602,0.374,0.295


## 9. Per-patient predictions

One CSV per cohort in `results/predictions/` (git-ignored), indexed by the original patient ID (`patient_id`), with `source_cohort`,
the clinical variables, `final_dgn`, `proba<MODEL>` / `prediction<MODEL>` for every model,
`n_positive_votes`, the ensemble `prediction` (2-of-3 vote of XGBoost, AdaBoost, CatBoost; the other
algorithms never enter it) and `outcome` (TP / FN / TN / FP of the ensemble).

In [10]:
SHORT_NAME = {
    "XGBoost": "XGB",
    "AdaBoost": "ADA",
    "CatBoost": "CAT",
    "LightGBM": "LGBM",
    "TabPFN": "TABPFN",
    "LogisticRegression": "LR",
    "RandomForest": "RF",
}


def ensemble_outcome(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    return np.select(
        [(y == 1) & (pred == 1), (y == 1) & (pred == 0), (y == 0) & (pred == 0)],
        ["TP", "FN", "TN"],
        default="FP",
    )


def save_cohort_predictions(X, y, source, models, thresholds, path, scores=None):
    """Per-patient CSV; `scores` overrides predict_proba (used for the out-of-fold pass)."""
    out = X.copy()
    out.insert(0, "source_cohort", source)
    out["final_dgn"] = y

    votes = []
    for name, model in models.items():
        proba = scores[name] if scores is not None else model.predict_proba(X)[:, 1]
        vote = (proba >= thresholds[name]).astype(int)
        out[f"proba{SHORT_NAME[name]}"] = proba
        out[f"prediction{SHORT_NAME[name]}"] = vote
        if name in MODEL_NAMES:
            votes.append(vote)

    out["n_positive_votes"] = np.vstack(votes).sum(axis=0)
    out["prediction"] = (out["n_positive_votes"] >= N_VOTES_POSITIVE).astype(int)
    out["outcome"] = ensemble_outcome(out["final_dgn"], out["prediction"])
    out.to_csv(path)

    c = out["outcome"].value_counts().reindex(["TP", "FN", "TN", "FP"], fill_value=0)
    print(
        f"  {path.name:<42} n={len(out):5}  TP={c.TP:4} FN={c.FN:3} TN={c.TN:5} FP={c.FP:5}  "
        f"Sens={c.TP / (c.TP + c.FN):.4f}  Spec={c.TN / (c.TN + c.FP):.4f}"
    )
    return out


r = results


def derivation_source(index):
    return np.where(index.isin(r["profundus_ids"]), "PROFUNDUS", "ADVISED")


print(f"Per-patient predictions (ensemble = {N_VOTES_POSITIVE}-of-{len(MODEL_NAMES)} "
      f"vote of {', '.join(MODEL_NAMES)})\n")

predictions = {
    "held-out test": save_cohort_predictions(
        r["X_test"], r["y_test"], derivation_source(r["X_test"].index),
        r["models"], r["thresholds"], PREDICTIONS_DIR / "predictions_heldout_test.csv"),
    "development (in-sample)": save_cohort_predictions(
        r["X_dev"], r["y_dev"], derivation_source(r["X_dev"].index),
        r["models"], r["thresholds"], PREDICTIONS_DIR / "predictions_development_insample.csv"),
    "development (out-of-fold)": save_cohort_predictions(
        r["X_dev"], r["y_dev"], derivation_source(r["X_dev"].index),
        r["models"], r["thresholds"], PREDICTIONS_DIR / "predictions_development_oof.csv",
        scores=r["development_oof"]),
    "external validation": save_cohort_predictions(
        r["X_ext"], r["y_ext"], "PRESC1SE",
        r["models"], r["thresholds"], PREDICTIONS_DIR / "predictions_external_validation.csv"),
}

derivation = pd.concat([
    predictions["development (out-of-fold)"].assign(split="development (out-of-fold)"),
    predictions["held-out test"].assign(split="held-out test"),
])
derivation.to_csv(PREDICTIONS_DIR / "predictions_derivation_cohort.csv")
c = derivation["outcome"].value_counts().reindex(["TP", "FN", "TN", "FP"], fill_value=0)
print(f"  {'predictions_derivation_cohort.csv':<42} n={len(derivation):5}  TP={c.TP:4} FN={c.FN:3} "
      f"TN={c.TN:5} FP={c.FP:5}  Sens={c.TP / (c.TP + c.FN):.4f}  Spec={c.TN / (c.TN + c.FP):.4f}")

misclassified = pd.concat([
    df.loc[df["outcome"].isin(["FN", "FP"])].assign(split=split)
    for split, df in predictions.items()
])
misclassified.insert(0, "split", misclassified.pop("split"))
misclassified.to_csv(PREDICTIONS_DIR / "misclassified_patients.csv")

print(f"\nWritten to {PREDICTIONS_DIR}")

Per-patient predictions (ensemble = 2-of-3 vote of XGBoost, AdaBoost, CatBoost)

  predictions_heldout_test.csv               n=  858  TP=  67 FN=  0 TN=  501 FP=  290  Sens=1.0000  Spec=0.6334


  predictions_development_insample.csv       n= 3431  TP= 265 FN=  4 TN= 2014 FP= 1148  Sens=0.9851  Spec=0.6369
  predictions_development_oof.csv            n= 3431  TP= 262 FN=  7 TN= 1994 FP= 1168  Sens=0.9740  Spec=0.6306


  predictions_external_validation.csv        n= 1282  TP=  28 FN=  0 TN= 1004 FP=  250  Sens=1.0000  Spec=0.8006
  predictions_derivation_cohort.csv          n= 4289  TP= 329 FN=  7 TN= 2495 FP= 1458  Sens=0.9792  Spec=0.6312

Written to ../results/predictions


## 10. Locked models for inference

Each model is saved as its complete `Pipeline` (mean imputation → standardisation → classifier),
and the fitted imputer and scaler on their own (identical inside every pipeline, checked before
saving). `manifest.json` holds the thresholds, the feature order, the voting rule and the two
transforms applied before the pipeline. The inference notebooks read exactly these files.

TabPFN keeps the training patients as its context, so its pipeline is patient-level data and is
written to `results/predictions/` (git-ignored) instead of `models/`.

In [11]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# TabPFN stores the training patients, so it is kept with the patient-level files.
for name, model in r["models"].items():
    joblib.dump(model, (PREDICTIONS_DIR if name == "TabPFN" else MODELS_DIR) / f"pipeline_{name}.joblib")

# Imputer and scaler are identical inside every pipeline.
ref = MODEL_NAMES[0]
imputer = r["models"][ref].named_steps["imputer"]
scaler = r["models"][ref].named_steps["scaler"]
for name in MODEL_NAMES[1:]:
    assert np.allclose(r["models"][name].named_steps["imputer"].statistics_, imputer.statistics_)
    assert np.allclose(r["models"][name].named_steps["scaler"].mean_, scaler.mean_)
    assert np.allclose(r["models"][name].named_steps["scaler"].scale_, scaler.scale_)
joblib.dump(imputer, MODELS_DIR / "imputer_final.joblib")
joblib.dump(scaler, MODELS_DIR / "scaler_final.joblib")

manifest = {
    "ensemble_members": list(MODEL_NAMES),
    "n_votes_positive": N_VOTES_POSITIVE,
    "thresholds": {k: float(v) for k, v in r["thresholds"].items()},
    "feature_order": list(r["X_dev"].columns),
    "fixed_zero_columns": list(FIXED_ZERO_COLS),
    "log2_transform": ["ddimer_value"],
}
with open(MODELS_DIR / "manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)

for p in sorted(MODELS_DIR.glob("*.joblib")) + [MODELS_DIR / "manifest.json"]:
    print(f"  {p.name:<30} {p.stat().st_size / 1024:7.1f} KB")

  imputer_final.joblib               1.2 KB
  pipeline_AdaBoost.joblib         128.5 KB
  pipeline_CatBoost.joblib          16.0 KB
  pipeline_LightGBM.joblib         881.7 KB
  pipeline_LogisticRegression.joblib     3.2 KB
  pipeline_RandomForest.joblib   12415.2 KB
  pipeline_XGBoost.joblib          295.0 KB
  scaler_final.joblib                1.3 KB
  manifest.json                      1.0 KB
